# Golden Set v3.1 + Blind VLM 통합 실험

이 노트북은 다음 두 개선만 검증합니다.

1. B15 정답 12→15, G22 선두 기권+근거 설명, 문서명 내부 대괄호 보존
2. 기존 검색 결과에서 시각 후보를 찾고 VLM으로 판독하되 정답 문서/page/bbox/hash는 입력하지 않음

검색기와 `ask_rfp_v9` 생성기 내부는 수정하지 않습니다. 실행 전 커널을 `myenv`로 선택하고 **Restart Kernel**한 뒤 첫 셀부터 순서대로 실행하세요.

In [ ]:
import os
from pathlib import Path

ROOT = Path('/home/kongseok/sprint-public-procurement-rag-assistant')
os.chdir(ROOT)
# KURE 검색 임베딩은 CPU에서 실행합니다. VLM 서버 프로세스만 GPU 0을 사용합니다.
os.environ['CUDA_VISIBLE_DEVICES'] = ''

archive_candidates = [
    ROOT / 'denoising-dirty-documents.zip',
    ROOT / 'denoising-dirty-documents.Zip',
    Path('/home/kongseok/denoising-dirty-documents.zip'),
    Path('/home/kongseok/denoising-dirty-documents.Zip'),
]
ARCHIVE_PATH = next((path for path in archive_candidates if path.exists()), None)
if ARCHIVE_PATH is None:
    raise FileNotFoundError('원본 압축 파일을 레포 루트 또는 /home/kongseok에 올려주세요.')
if not (ROOT / 'output/chunks.pkl').exists():
    raise FileNotFoundError('output/chunks.pkl이 없습니다.')
print('레포:', ROOT)
print('원본 압축:', ARCHIVE_PATH)

In [ ]:
import getpass

if not os.environ.get('OPENAI_API_KEY', '').strip():
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API Key: ').strip()
if not os.environ['OPENAI_API_KEY'].startswith('sk-'):
    raise ValueError('OpenAI API Key를 확인하세요.')
print('API 키가 메모리에 설정되었습니다. 출력하거나 파일에 저장하지 않습니다.')

In [ ]:
import json
import subprocess
import time
import urllib.request

VLM_URL = 'http://127.0.0.1:8003/v1'
VLM_MODEL = 'Qwen/Qwen3-VL-8B-Instruct'
vlm_process = None
vlm_log_handle = None

def server_models():
    try:
        with urllib.request.urlopen(VLM_URL + '/models', timeout=3) as response:
            return json.load(response)
    except Exception:
        return None

models = server_models()
if models is None:
    executable = Path('/home/kongseok/vllm-venv/bin/vllm')
    if not executable.exists():
        raise FileNotFoundError('/home/kongseok/vllm-venv/bin/vllm이 없습니다.')
    log_path = ROOT / 'output/vlm_blind_server.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    # 이전 실패 로그와 이번 실행을 섞지 않도록 매번 새 로그를 씁니다.
    vlm_log_handle = log_path.open('w', encoding='utf-8')
    # Jupyter의 myenv 환경을 넘기지 않고, 어제 성공한 vllm-venv 셸 환경을 그대로 사용합니다.
    shell_command = r'''
unset PYTHONPATH PYTHONHOME LD_PRELOAD
source /home/kongseok/vllm-venv/bin/activate
export CUDA_VISIBLE_DEVICES=0
export LD_LIBRARY_PATH="$VIRTUAL_ENV/lib/python3.12/site-packages/torch/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cu13/lib"
exec "$VIRTUAL_ENV/bin/vllm" serve Qwen/Qwen3-VL-8B-Instruct --host 127.0.0.1 --port 8003 --gpu-memory-utilization 0.85 --max-model-len 8192
'''
    command = ['/bin/bash', '-lc', shell_command]
    vlm_process = subprocess.Popen(
        command, cwd=ROOT, stdout=vlm_log_handle,
        stderr=subprocess.STDOUT, start_new_session=True,
    )
    print('VLM 서버 시작 중... 로그:', log_path)
    for _ in range(180):
        if vlm_process.poll() is not None:
            raise RuntimeError(f'VLM 서버가 종료됐습니다. 로그를 확인하세요: {log_path}')
        models = server_models()
        if models is not None:
            break
        time.sleep(5)
    else:
        raise TimeoutError(f'15분 안에 VLM 서버가 준비되지 않았습니다: {log_path}')
served = [item.get('id') for item in models.get('data', [])]
if VLM_MODEL not in served:
    raise RuntimeError(f'8003 포트의 모델이 다릅니다: {served}')
print('VLM 서버 준비 완료:', served)

In [ ]:
from datetime import datetime, timezone
from experiments.vlm_blind_v1.run_experiment import run

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
try:
    RESULT_DIR = run(ARCHIVE_PATH, RUN_ID)
finally:
    # 이 노트북이 직접 띄운 VLM 서버만 종료합니다. 기존 서버는 건드리지 않습니다.
    if vlm_process is not None and vlm_process.poll() is None:
        vlm_process.terminate()
        try:
            vlm_process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            vlm_process.kill()
    if vlm_log_handle is not None:
        vlm_log_handle.close()
print('완료:', RESULT_DIR)

In [ ]:
import pandas as pd

summary = json.loads((RESULT_DIR / 'summary.json').read_text(encoding='utf-8'))
display(pd.DataFrame([summary]).T.rename(columns={0: '결과'}))
print('gold locator 사용:', summary['gold_document_locator_used'])
print('채점기:', summary['scorer_version'])
print('골든셋 패치:', summary['golden_patch_version'])
print('결과 폴더:', RESULT_DIR)